# Entity ***Positions***
+ Layer **gold**
+ Priority **1**
---
### Imports

In [0]:
%run ../common/medallion_functions

### Parameters

In [0]:
layer = dbutils.widgets.get("layer")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

In [0]:
table_name = f"dim_{entity_name}"

### Execution

In [0]:
sv_positions_df = spark.read.table(f"uniswap.silver.{entity_name}")

In [0]:
gld_positions_df = spark.sql(f"""
    WITH cte_silver_positions AS (
        SELECT
            pk_position_id,
            tx_id,
            pool_id,
            token0_id,
            token1_id,
            tick_lower_id,
            tick_upper_id,
            owner_address,
            deposited_token0,
            deposited_token1,
            withdrawn_token0,
            withdrawn_token1,
            collected_fees_token0,
            collected_fees_token1,
            tx_timestamp,
            CAST(SPLIT(tick_upper_id, "#")[1] AS INT) AS tick_upper_idx,
            CAST(SPLIT(tick_lower_id, "#")[1] AS INT) AS tick_lower_idx
        FROM {{df}}
    ),
    cte_width_ticks AS (
        SELECT
            *,
            tick_upper_idx - tick_lower_idx AS position_width_ticks
        FROM cte_silver_positions
    ),
    cte_positions_width_percentiles AS (
        SELECT
            percentile(position_width_ticks, 0.25) AS lower_percentile,
            percentile(position_width_ticks, 0.75) AS wide_percentile
        FROM cte_width_ticks
    )
    SELECT
        positions.*,
        CASE
            WHEN position_width_ticks <= pct.lower_percentile THEN "narrow"
            WHEN position_width_ticks <= pct.wide_percentile THEN "medium"
            WHEN position_width_ticks > pct.wide_percentile THEN "wide"
            ELSE "unknown"
        END AS width_category,
        DATEDIFF(current_date(), tx_timestamp) AS position_age_days,
        CURRENT_TIMESTAMP() AS _load_timestamp_gold
    FROM cte_width_ticks AS positions
    CROSS JOIN cte_positions_width_percentiles AS pct
""", df = sv_positions_df)

### Save & exit

In [0]:
load_result = load_entity(gld_positions_df,
                        table_name,
                        load_pattern,
                        layer
                        )

In [0]:
dbutils.notebook.exit(load_result)